[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/scottyUX/CSE115A-Summer/blob/main/week-2/lab-02b-cursor-skills.ipynb)

# Lab 02b — Cursor Skills

## What You'll Learn

By the end of this lab you will be able to:

- Explain why skills exist and when to use them over rules.
- Create a basic instruction-only skill.
- Use asset files to keep skills focused and token-efficient.
- Apply few-shot examples as skill instructions.
- Delegate deterministic work to scripts inside a skill.
- Verify that Cursor has discovered your skills.

---

## Step 1 — Why Skills?

Before building anything, understand the problem skills solve.

### The Problem: Context Saturation

Every token in the agent's context window costs time and money. When you load an entire codebase, a full style guide, legal templates, and validation logic into every session, two things happen:

- **Context saturation** — the model loses focus on what matters.
- **Tool bloat** — the agent spends time reasoning about tools it doesn't need.

### The Solution: Progressive Disclosure

Skills implement progressive disclosure. The agent receives only lightweight metadata (name + description) upfront. The full instructions load only when the agent determines the skill is relevant to your request.

```
Without skills:  [full codebase] + [style guide] + [legal text] + [validation logic] → every session
With skills:     [metadata only] → agent loads what it needs, when it needs it
```

### Skills vs. Rules vs. MCP

| Tool | Analogy | When to use |
|---|---|---|
| **Rules** | Standing orders | Behavior that should always apply — style, safety, commit format |
| **Skills** | Specialist knowledge | Domain-specific tasks loaded on demand |
| **MCP servers** | Hands | External tools the agent can call — databases, APIs, browsers |

> Skills are the agent's brains for a specific domain. MCP servers are its hands. Rules are its standing orders.

---

## Step 2 — Core Concepts

### What a Skill Is

A skill is a **directory** containing a `SKILL.md` file and optional supporting assets. The folder name must match the `name` field in the frontmatter.

```
my-skill/
├── SKILL.md          required — definition and instructions
├── scripts/          optional — Python or Bash scripts
├── references/       optional — documentation, templates, legal text
└── assets/           optional — static files, data, examples
```

### Two Scopes

| Scope | Location | Applies to |
|---|---|---|
| **Project** | `.cursor/skills/` in the repo | This project only — version-controlled, shared with team |
| **User** | `~/.cursor/skills/` | All your projects — personal defaults |

### SKILL.md Format

```markdown
---
name: skill-name
description: When and why the agent should use this skill — be specific.
---

Your instructions here.
```

The `description` field is the routing mechanism. The agent reads it to decide whether this skill is relevant. A vague description means the skill never fires. A precise one means it fires exactly when you need it.

### Verify Discovery

After creating any skill, confirm Cursor found it:

1. Open **Cursor Settings** → **Rules**
2. Look for your skill name under the **Agent Decides** section

Or ask the agent directly:

```
What skills are currently available to you?
```

---

## Step 3 — Level 1: Basic Router

**Pattern:** Instruction-only. A single `SKILL.md` file, no supporting assets.

**When to use:** When the task requires judgment and reasoning, not external content. The instructions fit in a few lines.

---

### Skill: `git-commit-formatter`

This skill enforces the [Conventional Commits](https://www.conventionalcommits.org/) specification. Every commit message follows a consistent format that tools and humans can parse.

Create `.cursor/skills/git-commit-formatter/SKILL.md`:

```markdown
---
name: git-commit-formatter
description: Formats git commit messages using the Conventional Commits specification. Use when the user asks to commit changes or write a commit message.
---

Format all commit messages using the Conventional Commits specification:

  <type>(<optional scope>): <short summary>

Valid types: feat, fix, docs, test, refactor, chore, perf, ci

Rules:
- Summary must be lowercase and under 72 characters.
- Use the imperative mood: "add" not "added", "fix" not "fixed".
- If the change is breaking, append `!` after the type: `feat!:`
- Add a blank line and body paragraph for context when the change is non-obvious.

Examples:
  feat(grade-calc): add validate_scores function
  fix: handle empty score list in calculate_grade
  docs: add SKILL.md format reference to week-2 README
  test: add edge cases for grade boundary conditions
```

### Exercise

1. Create the file above in your repo.
2. Verify the skill appears in Cursor Settings → Rules.
3. In Agent chat, ask:

```
I just added validate_scores to grade_calculator.py. Write a commit message for this change.
```

The agent should produce a properly formatted Conventional Commit without you mentioning the format.

### Checkpoint 1
The agent produced a commit message matching the Conventional Commits format. The skill appears in Cursor Settings → Rules.

---

## Step 4 — Level 2: Asset Utilization

**Pattern:** The `SKILL.md` stays lightweight. Heavy or exact content lives in a `references/` file the skill points to.

**When to use:** When the skill needs to reproduce exact text — legal language, license headers, boilerplate — where paraphrasing would introduce errors or hallucinations.

---

### Skill: `license-header-adder`

This skill adds a standard license header to every new source file. The exact header text lives in a reference file — not in the instructions — so the agent copies it verbatim instead of generating it from memory.

**Directory structure:**
```
.cursor/skills/license-header-adder/
├── SKILL.md
└── references/
    └── HEADER_TEMPLATE.txt
```

**`references/HEADER_TEMPLATE.txt`:**
```
# CSE 115A Summer 2026
# Software Engineering in the Age of AI
#
# This file is part of the CSE 115A course materials.
# Unauthorized distribution outside of enrolled students is prohibited.
#
# Author: [student name]
# Date: [date]
```

**`SKILL.md`:**
```markdown
---
name: license-header-adder
description: Adds the course license header to new Python source files. Use when creating any new .py file.
paths:
  - "**/*.py"
---

When creating a new Python file, add the license header from
references/HEADER_TEMPLATE.txt as the first lines of the file.

Copy the header exactly — do not paraphrase or regenerate it.
Replace [student name] with the file author and [date] with today's date.
```

### Why reference files matter

If you put the license text directly in `SKILL.md`, two things go wrong:
1. The model may paraphrase legal language — subtle changes that matter legally.
2. The text consumes tokens in every session the skill loads.

Reference files solve both: the agent reads the file directly and copies it verbatim.

### Exercise

1. Create the directory structure and both files above.
2. Ask the agent to create a new Python file:

```
Create a new file week-2/utils.py with a function format_percentage(value: float) -> str
that returns a value formatted as a percentage string, e.g. 0.875 -> "87.5%".
```

3. Confirm the license header appears at the top of `utils.py`, copied exactly from the template.

### Checkpoint 2
`week-2/utils.py` exists with the license header at the top, copied verbatim from `HEADER_TEMPLATE.txt`.

---

## Step 5 — Level 3: Few-Shot Learning

**Pattern:** Instead of writing rules, show the agent examples of input and expected output. The agent learns the pattern from the examples.

**When to use:** When the transformation is hard to describe in words but easy to demonstrate. Conversion tasks, formatting tasks, and code generation with a specific output shape all benefit from examples over rules.

---

### Skill: `json-to-pydantic`

This skill converts JSON data structures into Python Pydantic models. Rather than writing rules like "infer types from values, use Optional for nullable fields," we show the agent exactly what input maps to what output.

**Directory structure:**
```
.cursor/skills/json-to-pydantic/
├── SKILL.md
└── examples/
    ├── student.json
    └── student_model.py
```

**`examples/student.json`:**
```json
{
  "name": "Alex Johnson",
  "student_id": "12345",
  "scores": [88, 92, 75],
  "enrolled": true,
  "advisor": null
}
```

**`examples/student_model.py`:**
```python
from pydantic import BaseModel
from typing import Optional


class Student(BaseModel):
    name: str
    student_id: str
    scores: list[float]
    enrolled: bool
    advisor: Optional[str] = None
```

**`SKILL.md`:**
```markdown
---
name: json-to-pydantic
description: Converts JSON data structures to Python Pydantic models. Use when given a JSON object or schema that needs a typed Python model.
---

Convert JSON to a Pydantic BaseModel following the pattern in examples/.

Reference examples/student.json → examples/student_model.py to understand
the expected input/output shape before converting new JSON.

Rules:
- Use `Optional[T] = None` for nullable fields.
- Use `list[T]` for arrays — infer element type from values.
- Class name should be PascalCase derived from the JSON structure's purpose.
- Always include the `from __future__ import annotations` import for forward refs.
```

### Exercise

1. Create the directory structure and all files above.
2. Ask the agent to convert this JSON:

```
Use the json-to-pydantic skill to convert this JSON to a Pydantic model
and save it to week-2/models.py:

{
  "course_id": "CSE115A",
  "title": "Software Engineering in the Age of AI",
  "credits": 4,
  "students": [],
  "instructor": null
}
```

3. Review `week-2/models.py` — confirm the model matches the pattern from the examples.

### Checkpoint 3
`week-2/models.py` contains a valid Pydantic model with correct types, `Optional` for nullable fields, and matches the pattern shown in the examples.

---

## Step 6 — Level 4: Procedural Scripts

**Pattern:** For validation and deterministic checks, delegate to a script instead of asking the model to reason. Scripts produce consistent, verifiable results. Models don't.

**When to use:** When correctness is binary (pass/fail), when the logic is complex enough that the model might make mistakes, or when you need an auditable result.

---

### Skill: `grade-validator`

This skill validates that a Python grade calculator module meets course standards. Instead of asking the agent to eyeball the code, it runs a script that checks mechanically.

**Directory structure:**
```
.cursor/skills/grade-validator/
├── SKILL.md
└── scripts/
    └── validate.py
```

**`scripts/validate.py`:**

In [ ]:
# .cursor/skills/grade-validator/scripts/validate.py
# Save this file in your repo at the path above — do not run it here in Colab

import ast
import sys

REQUIRED_FUNCTIONS = ["calculate_grade", "validate_scores"]

def validate(filepath: str) -> None:
    errors = []

    with open(filepath) as f:
        source = f.read()

    tree = ast.parse(source)
    defined = {node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)}

    for fn in REQUIRED_FUNCTIONS:
        if fn not in defined:
            errors.append(f"MISSING: function '{fn}' not found")

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            if not (node.returns or node.args.annotations):
                errors.append(f"MISSING ANNOTATIONS: '{node.name}' has no type annotations")
            if not (ast.get_docstring(node)):
                errors.append(f"MISSING DOCSTRING: '{node.name}' has no docstring")

    if errors:
        for e in errors:
            print(e)
        sys.exit(1)
    else:
        print("PASS: all checks passed")
        sys.exit(0)

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python validate.py <filepath>")
        sys.exit(1)
    validate(sys.argv[1])

**`SKILL.md`:**
```markdown
---
name: grade-validator
description: Validates a Python grade calculator file against course standards. Use when asked to validate or check grade_calculator.py.
---

Run the validation script — do not eyeball the code.

Steps:
1. Run: `python .cursor/skills/grade-validator/scripts/validate.py <filepath>`
2. If exit code is 0: report PASS and summarize what was checked.
3. If exit code is 1: report each error line, then fix the issues and re-run until all checks pass.

Checks performed by the script:
- Required functions exist: calculate_grade, validate_scores
- All functions have type annotations
- All functions have docstrings
```

### Why scripts beat model reasoning for validation

| Approach | Result |
|---|---|
| Ask the model to check for missing docstrings | May miss some, may hallucinate others |
| Run `validate.py` | Deterministic — same input always produces same output |

Scripts are auditable, version-controlled, and reliable. For pass/fail checks, always prefer a script.

### Exercise

1. Create the directory structure and both files above.
2. Ask the agent to validate your grade calculator:

```
/grade-validator validate week-1/grade_calculator.py
```

3. If the script reports failures, let the agent fix them and re-run until it passes.
4. Commit the passing file.

### Checkpoint 4
The validation script exits with code 0 for `grade_calculator.py`. All four skills are committed to `.cursor/skills/`.

---

## Step 7 — Choosing the Right Pattern

You now have four patterns. Here is how to choose:

| Pattern | Use when | Example |
|---|---|---|
| **Level 1 — Instruction only** | Task requires judgment, instructions are short | Commit formatting, code review style |
| **Level 2 — Asset utilization** | Exact text must be reproduced without paraphrasing | License headers, legal boilerplate, API specs |
| **Level 3 — Few-shot examples** | Transformation is hard to describe but easy to show | JSON-to-model, format conversion, scaffolding |
| **Level 4 — Procedural scripts** | Correctness is binary and must be auditable | Validation, linting, compliance checks |

### The description field is everything

Across all four patterns, the single most important field is `description`. It determines whether the agent activates the skill at all.

**Weak description** (skill rarely fires):
```
description: Helps with Python code.
```

**Strong description** (skill fires exactly when needed):
```
description: Validates a Python grade calculator file against course standards.
Use when asked to validate or check grade_calculator.py.
```

### Skills are team assets

Commit `.cursor/skills/` to your repo. Every teammate gets the same agent capabilities. Skills evolve with the project — when a standard changes, update the skill once and everyone benefits.

---

## Lab Completion Checklist

| # | Task | Done |
|---|---|---|
| 1 | `git-commit-formatter` skill created and verified in Cursor Settings | |
| 2 | Agent produced a Conventional Commit message without being told the format | |
| 3 | `license-header-adder` skill created with reference file | |
| 4 | New Python file created with license header copied verbatim | |
| 5 | `json-to-pydantic` skill created with examples folder | |
| 6 | `week-2/models.py` generated matching the example pattern | |
| 7 | `grade-validator` skill created with validation script | |
| 8 | `grade_calculator.py` passes all script checks | |
| 9 | All four skills committed to `.cursor/skills/` | |

---

## Reflection Questions

1. What is the difference between a skill and a rule? When would you use each?
2. Why is the `description` field more important than the instruction body?
3. For the `grade-validator` skill, why does the script approach produce more reliable results than asking the model to check the code directly?
4. Which of the four patterns would you use to enforce an API design standard across a team? Why?

---

## Resources

- [Cursor Skills Docs](https://cursor.com/docs/skills)
- [Agent Skills Standard](https://agentskills.io)
- [Conventional Commits](https://www.conventionalcommits.org/)
- [Week 2 README](./README.md)
- [Lab 02 — Cursor CLI](./lab-02-cursor-cli.ipynb)